[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/permutations_course.ipynb)

# Permutations and combinations full course: practice notebook

One practice question for every teaching part of the video, in the same order, and three cells where Python does the
counting for you: it lists every outfit, every arrangement and every choice, then counts them, so you can see the
formula and the list agree. Work each question out on paper first, then type your answer into the box and press
**check**. It tells you whether you're right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the cell runs
again by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Type whole numbers without commas or spaces: `650000`. You can also type the working: `6*5*4`,
`7!/(2!*3!)`, `10C3` for 10 choose 3, `8P3` for the permutations of 8 things 3 at a time. For several answers,
separate them with commas, in the order the question asks. You can also put your answer between the quotes in a
cell's last line, `practice(1, "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula from the course on two pages:
[permutations and combinations cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/permutations_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re
from math import comb, perm, factorial
from itertools import permutations, combinations, product
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, ToggleButtons, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SKY, MINT, GOLD, VIOLET = "#6fb6bb", "#a0c27c", "#e8c56b", "#b49ad6"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "font.size": 13, "axes.titlesize": 15, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
NAMES = {"binomial": sp.binomial, "ff": sp.ff, "factorial": sp.factorial}
SUPS, SUBS = "⁰¹²³⁴⁵⁶⁷⁸⁹", "₀₁₂₃₄₅₆₇₈₉"
def sup(n): return "".join(SUPS[int(d)] for d in str(n))
def sub(n): return "".join(SUBS[int(d)] for d in str(n))
def digits(s, table): return "".join(str(table.index(ch)) for ch in s)

def prep(s):
    """Typed symbols → Python: − × ÷, ⁸C₃ and 8C3 (choose), ⁸P₃ and 8P3 (arrange), 650 000 → 650000."""
    s = s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("·", "*")
    s = re.sub(f"([{SUPS}]+)\\s*([CcPp])\\s*([{SUBS}]+)", lambda m: f"{digits(m[1], SUPS)}{m[2]}{digits(m[3], SUBS)}", s)
    s = re.sub(r"(\d+)\s*[Cc]\s*(\d+)", r"binomial(\1,\2)", s)
    s = re.sub(r"(\d+)\s*[Pp]\s*(\d+)", r"ff(\1,\2)", s)
    return re.sub(r"(?<=\d) (?=\d{3}\b)", "", s)

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [parse_expr(prep(p), local_dict=NAMES, transformations=RULES) for p in s.split(",") if p.strip()]

def same(got, want, tol=1e-9):
    try: return abs(float(sp.N(sp.sympify(got) - want, 30))) < tol
    except (TypeError, ValueError): return False

# Each question: the question, the answers in order, the working, and the move that cracks each part (shown when that
# part is wrong).
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets and the signs."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = [same(g, w) for g, w in zip(got, want)]
            if all(ok): print("✓ right"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def count(things):
    """Count by listing: go through every item one by one."""
    return sum(1 for _ in things)

## 1 · The fundamental principle of counting

If one thing can be done in m ways, and after it a second thing in n ways, the two together can be done in m × n
ways: every first choice goes with every second choice. That's the multiplication principle, and it works for any
number of steps. If you do one thing *or* another, never both, the choices add instead: AND means multiply, OR means
add (as long as the two groups don't overlap). With a rule attached, such as "the first digit can't be zero", fill
the hardest place first.

The cell below lists every outfit you can make from some T-shirts and some trousers, then counts the list. Move the
sliders to change the number of T-shirts or trousers: the list grows, and the count always matches the
multiplication.

In [ ]:
SHIRTS = [("red", CORAL), ("blue", SKY), ("green", MINT), ("yellow", GOLD), ("purple", VIOLET), ("orange", "#e0a050")]
TROUSERS = [("black", "#0f1514"), ("grey", "#8c938f"), ("navy", "#3f5f93"), ("brown", "#8b5e3c"), ("white", "#f4f1ea")]
SHIRT = [(-0.15, 0.6), (0.15, 0.6), (0.45, 0.55), (0.66, 0.3), (0.5, 0.17), (0.32, 0.3), (0.32, -0.1), (-0.32, -0.1),
         (-0.32, 0.3), (-0.5, 0.17), (-0.66, 0.3), (-0.45, 0.55)]
LEGS = [(-0.3, -0.13), (0.3, -0.13), (0.33, -0.78), (0.06, -0.78), (0, -0.36), (-0.06, -0.78), (-0.33, -0.78)]

OUTFITS = list(product([s for s, _ in SHIRTS[:3]], [t for t, _ in TROUSERS[:2]]))
assert count(OUTFITS) == 6 == 3 * 2                                                   # the video: 3 × 2 = 6
assert count(product(range(4), range(3))) == 12 == 4 * 3                              # … and 4 × 3 = 12
assert all(count(product(range(a), range(b))) == a * b for a in range(1, 7) for b in range(1, 6))

def outfits(shirts=3, trousers=2):
    rows = list(product(SHIRTS[:shirts], TROUSERS[:trousers]))          # Python lists every outfit …
    n = count(rows)                                                     # … and counts the list
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.01, 0.02, 0.47, 0.86]); ax.axis("off"); ax.set_aspect("equal")
    L = max(shirts, trousers, 3)                                        # the grid fills the space it has
    ax.set_xlim(-0.6, L - 0.4); ax.set_ylim(-(L - 1) - 0.7, 0.75)
    fig.text(0.02, 0.93, "a row for each T-shirt, a column for each pair of trousers", color=DIM, fontsize=12)
    for k, ((s, sc), (t, tc)) in enumerate(rows):
        i, j = divmod(k, trousers)
        cx, cy = j * 1.0, -i * 1.0
        ax.add_patch(Polygon([(cx + 0.36 * a, cy + 0.36 * b + 0.1) for a, b in LEGS], fc=tc, ec=CHALK, lw=0.9))
        ax.add_patch(Polygon([(cx + 0.36 * a, cy + 0.36 * b + 0.1) for a, b in SHIRT], fc=sc, ec=CHALK, lw=0.9))
        ax.text(cx + 0.33, cy + 0.27, str(k + 1), color=DIM, fontsize=9)
    tp = fig.add_axes([0.5, 0.0, 0.5, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    tp.text(0.0, 0.93, "the list", color=DIM, fontsize=13)
    per = 10                                                            # ten to a column
    for k, ((s, sc), (t, _)) in enumerate(rows):
        col, row = divmod(k, per)
        tp.text(0.0 + 0.33 * col, 0.85 - 0.056 * row, f"{k + 1:>2}  {s} + {t}", color=sc, fontsize=11.5, family="monospace")
    tp.text(0.0, 0.15, f"Python counted the list: {n} outfits", color=CHALK, fontsize=16)
    tp.text(0.0, 0.05, f"{shirts} T-shirts × {trousers} trousers = {shirts * trousers}   ✓ the same",
            color=GOLD, fontsize=16, fontweight="bold")
    assert n == shirts * trousers
    plt.show()

In [ ]:
interact(outfits, shirts=IntSlider(value=3, min=1, max=6, description="T-shirts"),
         trousers=IntSlider(value=2, min=1, max=5, description="trousers"));

In [ ]:
LUNCH = [("wrap", k, d) for k in range(4) for d in range(5)] + [("soup", k, d) for k in range(3) for d in range(5)]
EVEN4 = [n for n in range(1000, 10000) if n % 2 == 0]
DIFF4 = [n for n in range(1000, 10000) if len(set(str(n))) == 4]
Q[1] = {"text": "A food stall in Nairobi sells 4 kinds of wrap and 3 kinds of soup. Lunch is one wrap or one soup, and "
                "then one of 5 drinks. How many different lunches are there? Then: how many four-digit numbers are "
                "even? And how many four-digit numbers have four different digits?",
        "answer": [count(LUNCH), count(EVEN4), count(DIFF4)],
        "moves": ["a wrap OR a soup: add first, 4 + 3. Then AND a drink: multiply by 5.",
                  "fill the hardest place first: the last digit must be 0, 2, 4, 6 or 8, and the first can't be 0.",
                  "the first digit has 9 choices (not 0); then 0 is back, but each digit used is gone: 9, then 8, then 7."],
        "working": ["Lunch: a wrap or a soup is 4 + 3 = 7 choices (OR adds). Then a drink: 7 × 5 = 35 lunches (AND multiplies).",
                    "Even four-digit numbers: the last digit is 0, 2, 4, 6 or 8 (5 choices); the first is 1–9 (9 choices);",
                    "the middle two are anything (10 each). So 9 × 10 × 10 × 5 = 4500.",
                    "Check: the four-digit numbers are 1000 to 9999, 9000 of them, and half are even: 4500.",
                    "Four different digits: the first is 1–9 (9 choices). The second can be 0 again, but not the first",
                    "digit: 9 choices. Then 8, then 7. So 9 × 9 × 8 × 7 = 4536."]}
assert Q[1]["answer"] == [(4 + 3) * 5, 9 * 10 * 10 * 5, 9 * 9 * 8 * 7] == [35, 4500, 4536]
practice(1)

## 2 · Permutations

A permutation is an arrangement in a definite order. n different things can be put in a row in n! = n × (n − 1) × …
× 2 × 1 ways (and 0! = 1). The number of ways to arrange r of them is ⁿPᵣ = n × (n − 1) × … (r numbers) = n!/(n − r)!.
If repeats are allowed, every place has all n choices: nʳ. If some things are alike, divide by the ways the alike
ones can swap: BANANA has 6!/(3! 2!) = 60 orders. And n people round a table can sit in (n − 1)! ways, because turning
everyone one seat along doesn't change who sits next to whom.

The cell below builds every arrangement with Python and counts them. Pick how many things and how many places, with
or without repeats, and it prints the list and the count beside the formula. They always agree. Try ten digits and
four places with no repeats.

In [ ]:
def arranged(n, r, repeats):
    """Every arrangement of r of the digits 0, 1, … (n of them), listed by Python."""
    things = "0123456789"[:n]
    return ["".join(p) for p in (product(things, repeat=r) if repeats else permutations(things, r))]

assert count(arranged(10, 4, False)) == 5040 == perm(10, 4) == 10 * 9 * 8 * 7        # the video's PIN, no repeats
assert count(arranged(10, 4, True)) == 10000 == 10 ** 4                               # … and with repeats
assert count(arranged(5, 3, True)) == 125 and count(arranged(5, 3, False)) == 60      # five letters, three places
assert all(count(arranged(n, r, False)) == perm(n, r) == factorial(n) // factorial(n - r)
           for n in range(1, 8) for r in range(0, n + 1))

def formula(n, r, repeats):
    if repeats:
        return f"{n}{sup(r)} = " + " × ".join([str(n)] * r) + f" = {n ** r}" if r > 1 else f"{n}{sup(r)} = {n ** r}"
    if r > n: return f"{sup(n)}P{sub(r)}: there's nothing left to fill place {n + 1}, so 0"
    return f"{sup(n)}P{sub(r)} = " + " × ".join(str(n - k) for k in range(r)) + f" = {perm(n, r)}"

def arrangements(n=4, r=2, repeats="no repeats"):
    rep = repeats == "repeats allowed"
    rows = arranged(n, r, rep)                                          # Python lists them all …
    got = count(rows)                                                   # … and counts the list
    want = n ** r if rep else perm(n, r)
    fig = plt.figure(figsize=(11.5, 5.0))
    lp = fig.add_axes([0.0, 0.0, 0.56, 1.0]); lp.axis("off"); lp.set_xlim(0, 1); lp.set_ylim(0, 1)
    lp.text(0.02, 0.94, f"every arrangement ({'repeats allowed' if rep else 'no repeats'})", color=DIM, fontsize=13)
    cols = max(4, min(12, int(46 // (r + 2))))                          # as many to a line as fit
    lines = 14
    show = rows if len(rows) <= cols * lines else rows[:cols * (lines - 2)] + ["…"] + rows[-cols:]
    for k, w in enumerate(show):
        if w == "…":
            lp.text(0.02, 0.86 - 0.056 * (lines - 2), f"… {got - cols * (lines - 1)} more …", color=DIM, fontsize=12,
                    family="monospace"); continue
        k2 = k if k < cols * (lines - 2) else k - 1 + cols              # the last line sits under the "… more …" line
        row, col = divmod(k2, cols)
        lp.text(0.02 + col * 0.96 / cols, 0.86 - 0.056 * row, w, color=CHALK, fontsize=12, family="monospace")
    if not rows: lp.text(0.02, 0.8, "none", color=DIM, fontsize=14)
    tp = fig.add_axes([0.58, 0.0, 0.42, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    tp.text(0.0, 0.88, "things:  " + " ".join("0123456789"[:n]), color=SKY, fontsize=15, family="monospace")
    tp.text(0.0, 0.79, f"places:  {r}", color=MINT, fontsize=15, family="monospace")
    tp.text(0.0, 0.63, f"Python counted: {got}", color=CHALK, fontsize=19)
    tp.text(0.0, 0.52, "the formula:", color=DIM, fontsize=14)
    tp.text(0.0, 0.43, formula(n, r, rep), color=CORAL, fontsize=17 if len(formula(n, r, rep)) < 30 else 13,
            fontweight="bold")
    tp.text(0.0, 0.29, "✓ they agree" if got == want else "✗ they don't agree", color=GOLD, fontsize=20, fontweight="bold")
    if not rep and r <= n and r == n: tp.text(0.0, 0.18, f"all {n} in a row: {n}! = {factorial(n)}", color=DIM, fontsize=14)
    plt.show()

In [ ]:
interact(arrangements, n=IntSlider(value=4, min=1, max=10, description="things"),
         r=IntSlider(value=2, min=1, max=5, description="places"),
         repeats=ToggleButtons(options=["no repeats", "repeats allowed"], description=" "));

In [ ]:
TORONTO = set(permutations("TORONTO"))
ROUND7 = set()
for p in permutations("ABCDEFG"):                                       # seatings round a table, up to turning
    k = p.index("A"); ROUND7.add(p[k:] + p[:k])
Q[2] = {"text": "Six swimmers race at a school gala in Sydney. In how many ways can the gold, silver and bronze "
                "medals be won? Then: how many different orders are there of the letters of TORONTO? And in how many "
                "ways can seven friends sit round a circular table, if only who sits next to whom matters?",
        "answer": [count(permutations(range(6), 3)), count(TORONTO), count(ROUND7)],
        "moves": ["order matters (gold isn't silver): 6 choices, then 5, then 4.",
                  "7 letters, with two T's and three O's alike: 7! ÷ (2! × 3!).",
                  "round a table, fix one friend's seat and arrange the other six: (7 − 1)!."],
        "working": ["Medals: gold can go to any of 6, silver to any of the 5 left, bronze to any of the 4 left.",
                    "So ⁶P₃ = 6 × 5 × 4 = 120.",
                    "TORONTO: 7 letters, T twice, O three times. 7! = 5040 orders if every letter were different;",
                    "swapping the T's (2! ways) or the O's (3! ways) changes nothing, so 5040 ÷ (2 × 6) = 420.",
                    "Round the table: turning everyone one seat along gives the same seating, so fix one friend",
                    "and arrange the other six in the six seats left: 6! = 720. (Check: 7! ÷ 7 = 5040 ÷ 7 = 720.)"]}
assert Q[2]["answer"] == [perm(6, 3), factorial(7) // (factorial(2) * factorial(3)), factorial(6)] == [120, 420, 720]
practice(2)

## 3 · Combinations

A combination is a choice where order doesn't matter: a team, a hand of cards, the toppings on a pizza. Every group of
r things can be put in order in r! ways, so ⁿCᵣ = ⁿPᵣ / r! = n! / (r! (n − r)!). Choosing r to take is the same as
choosing n − r to leave, so ⁿCᵣ = ⁿCₙ₋ᵣ. Pascal's rule, ⁿCᵣ₋₁ + ⁿCᵣ = ⁽ⁿ⁺¹⁾Cᵣ, builds Pascal's triangle: each number
is the sum of the two above it, and row n holds ⁿC₀, ⁿC₁, … ⁿCₙ. The numbers in row n add up to 2ⁿ, the number of
subsets of n things.

The cell below draws Pascal's triangle. Move the **rows** slider to add rows: each new number appears as the sum of
the two above it. Pick a number with the other two sliders, **n** (its row) and **r** (its place in the row, counting
from 0): it shows which ⁿCᵣ that number is, and lists the choices it counts.

In [ ]:
def triangle(rows):
    """Pascal's triangle, built by adding: each number is the sum of the two above it."""
    t = [[1]]
    for _ in range(rows):
        prev = t[-1]
        t.append([1] + [prev[k - 1] + prev[k] for k in range(1, len(prev))] + [1])
    return t

T = triangle(10)
assert all(T[n][r] == comb(n, r) for n in range(11) for r in range(n + 1))          # the triangle is n choose r
assert T[7] == [1, 7, 21, 35, 35, 21, 7, 1] and sum(T[4]) == 16 == 2 ** 4           # the video's row 7, and 2⁴
assert all(sum(T[n]) == 2 ** n for n in range(11))
FRIENDS = "ABCDEFGHIJ"

def pascal(rows=5, n=4, r=2):
    r = min(r, n); t = triangle(rows)
    fig = plt.figure(figsize=(11.5, 5.4))
    ax = fig.add_axes([0.0, 0.08, 0.55, 0.86]); ax.axis("off")
    dy = 1.0; fs = 17 if rows <= 5 else 15 if rows <= 7 else 12.5 if rows <= 9 else 11
    H = max(rows, 5)                                                    # the triangle fills the space it has
    ax.set_xlim(-(H * 0.55 + 0.8), H * 0.55 + 0.8); ax.set_ylim(-H - 0.6, 0.7)
    pos = lambda m, k: ((k - m / 2) * 1.1, -m * dy)
    for m in range(1, rows + 1):                                        # the joins: each number to the two above it
        for k in range(m + 1):
            for pk in (k - 1, k):
                if 0 <= pk <= m - 1:
                    new = m == rows
                    (x0, y0), (x1, y1) = pos(m - 1, pk), pos(m, k)
                    ax.plot([x0, x1], [y0 - 0.22, y1 + 0.25], color=GOLD if new else GHOST, lw=1.6 if new else 1.0,
                            alpha=0.75 if new else 1)
    for m, row in enumerate(t):
        for k, v in enumerate(row):
            x_, y_ = pos(m, k)
            pick = (m, k) == (n, r)
            if pick: ax.plot([x_], [y_], "o", ms=34 if v < 100 else 40, mfc=SLATE2, mec=CORAL, mew=2.8, zorder=3)
            ax.text(x_, y_, str(v), ha="center", va="center", fontsize=fs + (2 if pick else 0), zorder=4,
                    color=CORAL if pick else GOLD if m == rows else CHALK, fontweight="bold" if pick else "normal")
    if rows >= 2:
        sums = [f"{t[rows - 1][k - 1]} + {t[rows - 1][k]} = {t[rows][k]}" for k in range(1, rows)]
        fig.text(0.02, 0.03, f"new row {rows}:  1,  " + ",  ".join(sums[:3]) + (",  …" if len(sums) > 3 else ",  1"),
                 color=GOLD, fontsize=13)
    tp = fig.add_axes([0.57, 0.0, 0.43, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    v = t[n][r]
    tp.text(0.0, 0.88, f"row {n}, place {r}:  {sup(n)}C{sub(r)} = {v}", color=CORAL, fontsize=21, fontweight="bold")
    who = FRIENDS[:n]
    names = "" if not n else f" {who}" if n == 1 else f" {', '.join(who)}" if n <= 5 else f" {who[0]} to {who[-1]}"
    tp.text(0.0, 0.79, f"the ways to choose {r} of {n} friend{'' if n == 1 else 's'}{names}", color=DIM, fontsize=13)
    picks = ["".join(c) for c in combinations(who, r)]
    assert len(picks) == v == comb(n, r)                                # the list and the triangle agree
    cols, lines = max(3, min(9, int(34 // (r + 2)))), 11
    shown = picks if len(picks) <= cols * lines else picks[:cols * (lines - 1)]
    for k, w in enumerate(shown):
        row_, col = divmod(k, cols)
        tp.text(0.0 + col * 0.98 / cols, 0.70 - 0.055 * row_, w if w else "(choose nobody)", color=CHALK, fontsize=12,
                family="monospace")
    if len(shown) < len(picks):
        tp.text(0.0, 0.70 - 0.055 * (lines - 1), f"… and {len(picks) - len(shown)} more", color=DIM, fontsize=12)
    tp.text(0.0, 0.03, f"Python listed {len(picks)} choices ✓", color=GOLD, fontsize=16, fontweight="bold")
    plt.show()

ROWS = IntSlider(value=5, min=0, max=10, description="rows")
N = IntSlider(value=4, min=0, max=5, description="n")
R = IntSlider(value=2, min=0, max=4, description="r")
ROWS.observe(lambda ch: setattr(N, "max", ch["new"]), names="value")    # you can only pick a row that's drawn …
N.observe(lambda ch: setattr(R, "max", ch["new"]), names="value")       # … and a place that's in that row

In [ ]:
interact(pascal, rows=ROWS, n=N, r=R);

In [ ]:
SMOOTHIES = [s for k in range(1, 8) for s in combinations(range(7), k)]
Q[3] = {"text": "A book club in Toronto picks 3 of 10 novels to read this season. How many different picks are "
                "there? Then: 10C3 = 120 and 10C4 = 210. Use Pascal's rule to find 11C4. And a juice bar in Mexico "
                "City has 7 fruits: how many different smoothies can you make with at least one fruit?",
        "answer": [count(combinations(range(10), 3)), comb(10, 3) + comb(10, 4), count(SMOOTHIES)],
        "moves": ["order doesn't matter for a reading list: 10P3 ÷ 3!.",
                  "Pascal's rule: the two numbers next to each other in row 10 add up to the one below them.",
                  "each fruit is a yes or a no, which gives 2⁷ choices, but one of them has no fruit at all."],
        "working": ["The same three novels in any order are the same pick, so it's a combination:",
                    "¹⁰C₃ = (10 × 9 × 8) ÷ (3 × 2 × 1) = 720 ÷ 6 = 120.",
                    "Pascal's rule: ¹⁰C₃ + ¹⁰C₄ = ¹¹C₄, so ¹¹C₄ = 120 + 210 = 330.",
                    "Check: ¹¹C₄ = (11 × 10 × 9 × 8) ÷ 24 = 7920 ÷ 24 = 330.",
                    "Smoothies: each of the 7 fruits is in or out, 2 × 2 × … × 2 = 2⁷ = 128 choices.",
                    "One of them is the empty cup, so 128 − 1 = 127 smoothies.",
                    "(Row 7 of Pascal's triangle adds up to 128 too: 1 + 7 + 21 + 35 + 35 + 21 + 7 + 1.)"]}
assert Q[3]["answer"] == [120, 330, 127] and comb(11, 4) == 330 and 2 ** 7 - 1 == 127
practice(3)

## 4 · Permutations and combinations together

Many questions need both: first choose, then arrange, and multiply the two counts. Choose a committee from two groups
by choosing from each and multiplying. For "at least one", count everything and subtract the count of "none". Shapes
count the same way: any 2 of n points make a line and any 3 (none of them in a line) make a triangle; an n-sided
shape has ⁿC₂ − n diagonals, because the n sides are among the ⁿC₂ lines.

In [ ]:
VOWELS, CONSONANTS = "EUAIO", "DCTN"                                    # the letters of EDUCATION
WORDS = [w for v in combinations(VOWELS, 2) for c in combinations(CONSONANTS, 3) for w in permutations(v + c)]
Q[4] = {"text": "A town council in Ghana forms a committee of 3 women and 2 men from 6 women and 5 men. How many "
                "committees are possible? Then: from the letters of EDUCATION, how many five-letter words (any "
                "string of letters, no letter used twice) have 2 vowels and 3 consonants? And how many diagonals "
                "does a 12-sided shape have?",
        "answer": [comb(6, 3) * comb(5, 2), count(WORDS), comb(12, 2) - 12],
        "moves": ["choose the women and the men separately, then multiply: 6C3 × 5C2.",
                  "choose 2 of the 5 vowels and 3 of the 4 consonants, then arrange the five letters: × 5!.",
                  "12C2 lines join two corners; 12 of them are sides."],
        "working": ["Women: ⁶C₃ = 20 ways. Men: ⁵C₂ = 10 ways. Both: 20 × 10 = 200 committees.",
                    "EDUCATION has vowels E, U, A, I, O (5) and consonants D, C, T, N (4).",
                    "Choose the vowels: ⁵C₂ = 10. Choose the consonants: ⁴C₃ = 4. Then arrange the five letters: 5! = 120.",
                    "So 10 × 4 × 120 = 4800 words.",
                    "Diagonals: any two of the 12 corners make a line, ¹²C₂ = 66 lines. 12 of them are sides,",
                    "so 66 − 12 = 54 diagonals. (Check: each corner joins 9 others by a diagonal: 12 × 9 ÷ 2 = 54.)"]}
assert Q[4]["answer"] == [200, 4800, 54] == [20 * 10, 10 * 4 * 120, 12 * 9 // 2]
practice(4)

## Sources
NCERT Mathematics Class XI, ch. 6 (Permutations and Combinations); OpenStax, Precalculus 2e, 11.5 (Counting
Principles). Both retrieved 2026-11-01. Every question is our own.